# Cross-compile for embedded: Cortex-A vs Cortex-M

The same controller has to run on very different chips — a Linux-class **Cortex-A**
(a Raspberry Pi 5) and a bare-metal **Cortex-M** microcontroller are worlds apart
in memory, real-time guarantees, and even *which math is cheap*. jaxility makes the
target a piece of **data**, not a code path: a `Target` profile describes the chip,
and the build planner + lowering adapt to it.

This tutorial compares two real target profiles, shows the **op-coverage contract**
(what lowers to embedded C and what doesn't — loudly), and explains how one model
reaches many chips.

In [1]:
import os, platform, ctypes
from pathlib import Path
def _preload_acados():  # macOS SIP strips DYLD_LIBRARY_PATH; preload acados libs by path
    root=os.environ.get("ACADOS_SOURCE_DIR")
    if not root or platform.system()!="Darwin": return
    for n in ("libblasfeo.dylib","libhpipm.dylib","libqpOASES_e.dylib","libacados.dylib"):
        p=Path(root)/"lib"/n
        if p.exists():
            try: ctypes.CDLL(str(p))
            except OSError: pass
_preload_acados()
import jax.numpy as jnp
from jaxility.targets import PI5, CORTEX_M4, NEOVERSE_N1, current_host_target
print("host target:", current_host_target().name)

host target: host-darwin


## Step 1 — Targets are data

A `Target` is a Pydantic record: ISA family, vector extensions, NPU, a memory
budget, and a real-time class. Adding a target is filling out the model — no
subclassing. Here are two ends of the spectrum side by side.

In [2]:
def describe(t):
    mem = t.memory
    return {
        "family": t.family,
        "vector": [v.value for v in t.vector_extensions],
        "MMU": mem.has_mmu,
        "code": f"{mem.code_bytes//1024} KiB" if mem.code_bytes < 2**30 else f"{mem.code_bytes//2**30} GiB",
        "data": f"{mem.data_bytes//1024} KiB" if mem.data_bytes < 2**30 else f"{mem.data_bytes//2**30} GiB",
        "realtime": f"{t.realtime.kind.value} / {t.realtime.scheduler.value}",
        "cycle_us": t.realtime.target_cycle_us,
    }
import json
for t in (PI5, CORTEX_M4):
    print(f"{t.name:10s}", json.dumps(describe(t)))

pi5        {"family": "cortex-a76", "vector": ["neon"], "MMU": true, "code": "8 GiB", "data": "8 GiB", "realtime": "soft / preempt-rt", "cycle_us": 1000}
cortex-m4  {"family": "cortex-m4", "vector": ["none"], "MMU": false, "code": "1024 KiB", "data": "192 KiB", "realtime": "hard / cyclic-executive", "cycle_us": 500}


Read that table top to bottom and the engineering reality jumps out:

| | **Pi 5 (Cortex-A76)** | **Cortex-M4** |
|---|---|---|
| memory | gigabytes, **MMU** | kilobytes, **no MMU** |
| real-time | **soft** (PREEMPT-RT) | **hard** (cyclic executive) |
| math | NEON SIMD, fast `float64` | single-precision FPU; `float64` is software-emulated, ~10× slower |
| allocation | dynamic OK | **no malloc after init** |

Same controller spec, two profiles — and the build planner and coverage table use
these fields to decide what's even *possible* on each chip.

## Step 2 — Quirks: the chip's gotchas are documented, not folklore

Each profile carries `quirks` — the hardware contracts a deployer must respect.
They surface in the build log so nothing is a surprise on silicon.

In [3]:
for t in (PI5, CORTEX_M4):
    print(f"== {t.name} ==")
    for q in t.quirks:
        print(f"  - {q.id}: {q.description[:96]}…")
    print()

== pi5 ==
  - pi5-4gb-sku: The Pi 5 ships in 4 GB / 8 GB / 16 GB skus. This profile pins the 8 GB baseline; the 4 GB sku ha…
  - preempt-rt-soft-not-hard: Pi 5 + PREEMPT_RT-patched Raspberry Pi OS reaches 1 kHz reliably with ~70 µs worst-case jitter m…
  - gicv2-irq-priority-needs-tuning: Pi 5's GIC-400 (GICv2) needs explicit IRQ-priority tuning for PREEMPT_RT to give the Jaxility co…
  - d-cache-clean-required-for-codegen-buffers: Generated controller code may live in a buffer allocated dynamically at deploy time (the ``jaxil…

== cortex-m4 ==
  - single-precision-fpu-only: Cortex-M4 has single-precision FPU (FPv4-SP-D16). Double-precision math is software-emulated and…
  - no-dynamic-allocation-after-init: Bare-metal M-class enforces no-malloc-after-init (PATTERNS §4.1).…



## Step 3 — The coverage contract: what lowers to embedded C

Not every JAX op can become embedded C. jaxility keeps an explicit **coverage
table** keyed by `(op, dtype, target_family)`. A supported op has a grade and an
implementation hint; an unsupported op is a **loud error**, never a silent
mis-lowering. The subset is broad — arithmetic, transcendentals, `matmul`, and
even bounded control flow (`lax.while_loop`, `lax.cond`, `jnp.where`,
`dynamic_slice`) — but it is *finite*. The Cartpole's smooth closed-form dynamics
lie inside it; the full articulated-arm MJX pipeline reaches for ops *outside* it,
which is why it doesn't lower to a single embedded controller today.

In [4]:
from jaxility.lowering.coverage import COVERAGE_TABLE
ops = sorted({op for (op, dt, fam) in COVERAGE_TABLE})
print(f"coverage table: {len(COVERAGE_TABLE)} rows across {len(ops)} distinct ops")
print("supported ops include:", ", ".join(ops[:16]))

coverage table: 60 rows across 20 distinct ops
supported ops include: add, div, dynamic_shape, dynamic_slice[static], dynamic_slice[traced], jnp.cos, jnp.exp, jnp.log, jnp.sin, jnp.sqrt, jnp.tan, jnp.where[static], jnp.where[traced], lax.cond[traced], lax.while_loop, matmul


## Step 4 — Supported vs unsupported, demonstrated

We translate two functions: one in the supported subset (lowers), one that reaches
for an op *outside* the table — a raw comparison — and is rejected with a
structured `CoverageError` naming the offending op.

In [5]:
from jaxility.lowering import translate
from jaxility.errors import CoverageError

def smooth(x, u):       # the kind of math a controller dynamics needs
    return jnp.array([x[1], jnp.sin(x[0]) + jnp.exp(-x[2]) * u[0], x[3], x[2]])

def with_branch(x, u):  # the raw '>' comparison op is outside the coverage table
    return jnp.where(x[0] > 0, x, -x)

try:
    translate(smooth, in_shapes=((4,),(1,)), name="smooth_ok")
    print("smooth dynamics      -> lowered OK (in the supported subset)")
except CoverageError as e:
    print("smooth dynamics      -> CoverageError:", e)

try:
    translate(with_branch, in_shapes=((4,),(1,)), name="branch_bad")
    print("control-flow dynamics-> lowered OK")
except CoverageError as e:
    print("control-flow dynamics-> CoverageError (caught loudly):", str(e)[:80], "…")

smooth dynamics      -> lowered OK (in the supported subset)
control-flow dynamics-> CoverageError (caught loudly): JAX primitive 'gt' has no handler [op='gt', dtype='float64', target_fami …


## Step 5 — One model, many chips

Building for a target is one call — `build_for_target(dynamics, spec, target=…)`.
The host build (below) runs the full lowering pipeline locally; the Cortex-A76
build is the *same call* with `target=PI5`, cross-compiled with the pinned Arm GNU
toolchain (that's the on-Pi run in tutorial #1). A Cortex-M target swaps the
profile for `arm-none-eabi`, bare-metal, single-precision. The model and the spec
don't change — only the `Target`.

In [6]:
import tempfile
import jaxterity.zoo as zoo
from jaxterity.zoo.cartpole import reduced_params
from jaxility.templates import lqr
from jaxility.builder import build_for_target
from pathlib import Path

robot = zoo.load("cartpole").with_provenance(("xc-tut","v0","tel"), calibrated=True)
p = reduced_params(robot)
def dyn(s,u):
    g,mp,mc,L=p["g"],p["mp"],p["mc"],p["L"]; th,xd,thd=s[1],s[2],s[3]
    si,co=jnp.sin(th),jnp.cos(th); den=mc+mp*si*si
    return jnp.array([xd,thd,(u[0]+mp*si*(L*thd**2+g*co))/den,
        (-u[0]*co-mp*L*thd**2*co*si-(mc+mp)*g*si)/(L*den)])
cf = translate(dyn, in_shapes=((4,),(1,)), name="xc_cartpole")
spec = lqr(cf, Q=(10.,10.,1.,1.), R=(0.1,), initial_state=(0.3,0.,0.,0.),
           input_bounds=((-20.,),(20.,)), name="xc_cartpole", horizon_steps=20, time_horizon_s=1.0)
host = current_host_target()
bundle = build_for_target(dynamics=cf, spec=spec, target=host,
    source_attestation_handle=bytes.fromhex(robot.attestation_handle),
    work_dir=Path(tempfile.mkdtemp()))
print(f"built for target '{host.name}' (family {host.family}); artifact",
      bundle.artifact.content_hash.hex()[:16], "…")
print(f"the SAME call with target=PI5 cross-compiles for the Cortex-A76 (tutorial #1);")
print(f"target=CORTEX_M4 targets bare-metal arm-none-eabi. The spec is unchanged —")
print(f"only the Target profile differs. (Cross-builds need the pinned Arm toolchain.)")

built for target 'host-darwin' (family host-darwin); artifact 66114092961b30ad …
the SAME call with target=PI5 cross-compiles for the Cortex-A76 (tutorial #1);
target=CORTEX_M4 targets bare-metal arm-none-eabi. The spec is unchanged —
only the Target profile differs. (Cross-builds need the pinned Arm toolchain.)


## Recap

Cross-compilation in jaxility is **target-as-data**:

- a `Target` profile captures the chip — ISA, vectors, NPU, memory budget,
  real-time class — and its quirks, so the build planner adapts instead of
  branching on chip names;
- the **coverage table** is the contract for what lowers to embedded C, with loud
  failures (a `CoverageError` names the op) — no silent mis-lowering;
- and one model + one spec reaches many chips: a Linux-class Cortex-A (Pi 5), a
  bare-metal Cortex-M, an NPU part — by swapping the `Target`, not the model.

That's how "one model, one truth" survives the jump from a workstation to a
microcontroller. Next in the series: **the predictive self-check co-processor** —
running a per-unit twin beside the controller to catch overheating and brownout
before they happen.